In [1]:
#1.保存训练好的CNN模型(.pt/.pth)
# 保存训练好的CNN模型(.pth)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1, 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# MNIST数据预处理（训练标准归一化）

transform = transforms.Compose([
    # 随机旋转、平移、缩放，大幅提升对极简竖线的兼容
    transforms.RandomAffine(degrees=12, translate=(0.12, 0.12), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# 模型、损失、优化器
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练15轮，每轮输出loss和测试准确率
epochs = 15
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # 测试集评估
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labs in test_loader:
            imgs, labs = imgs.to(device), labs.to(device)
            out = model(imgs)
            pred = torch.argmax(out, dim=1)
            total += labs.size(0)
            correct += (pred == labs).sum().item()
    test_acc = correct / total

    print(f"Epoch{epoch+1:2d} | Loss:{running_loss/len(train_loader):.4f} | Test Acc:{test_acc:.4f}")

# 保存权重
torch.save(model.state_dict(), "mnist_cnn.pth")
print("模型已保存为 mnist_cnn.pth")


100%|██████████| 9.91M/9.91M [00:05<00:00, 1.92MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 109kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.00MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 874kB/s]


Epoch 1 | Loss:0.2801 | Test Acc:0.9586
Epoch 2 | Loss:0.0998 | Test Acc:0.9763
Epoch 3 | Loss:0.0751 | Test Acc:0.9798
Epoch 4 | Loss:0.0659 | Test Acc:0.9829
Epoch 5 | Loss:0.0580 | Test Acc:0.9845
Epoch 6 | Loss:0.0553 | Test Acc:0.9856
Epoch 7 | Loss:0.0508 | Test Acc:0.9824
Epoch 8 | Loss:0.0478 | Test Acc:0.9840
Epoch 9 | Loss:0.0474 | Test Acc:0.9858
Epoch10 | Loss:0.0436 | Test Acc:0.9873
Epoch11 | Loss:0.0428 | Test Acc:0.9880
Epoch12 | Loss:0.0419 | Test Acc:0.9861
Epoch13 | Loss:0.0361 | Test Acc:0.9856
Epoch14 | Loss:0.0386 | Test Acc:0.9869
Epoch15 | Loss:0.0357 | Test Acc:0.9878
模型已保存为 mnist_cnn.pth


In [4]:
#%%writefile app.py
#1.写FastAPI服务:加载模型,定义请求/响应格式。实现/predict端点
import torch
import io
import uvicorn
import nest_asyncio
import torch.nn as nn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from PIL import Image
import numpy as np

# 解决Jupyter Notebook事件循环冲突
nest_asyncio.apply()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 和训练完全一致的网络结构
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1, 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 加载模型权重
model = SimpleCNN().to(device)
model.load_state_dict(torch.load("mnist_cnn.pth", map_location=device))
model.eval()
print("模型加载成功")

app = FastAPI(title="MNIST 手写数字识别API")

# 完整预处理：裁剪居中 + 黑白反转 + 归一化标准化（和训练完全对齐）
def backend_preprocess(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('L')
    gray_img = np.array(img, dtype=np.uint8)

    # 1.二值化过滤杂点
    threshold = 30
    binary = np.where(gray_img > threshold, 255, 0).astype(np.uint8)

    # 2.裁剪有效数字区域
    y_coords, x_coords = np.where(binary == 255)
    if len(y_coords) == 0:
        raise ValueError("图片未检测到数字")
    min_y, max_y = y_coords.min(), y_coords.max()
    min_x, max_x = x_coords.min(), x_coords.max()
    crop = binary[min_y:max_y+1, min_x:max_x+1]

    # 3.填充正方形、数字居中
    h, w = crop.shape
    max_side = max(h, w)
    pad_top = (max_side - h) // 2
    pad_bottom = max_side - h - pad_top
    pad_left = (max_side - w) // 2
    pad_right = max_side - w - pad_left
    square = np.pad(crop, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant", constant_values=0)

    # 4.缩放至28×28
    pil_square = Image.fromarray(square).resize((28, 28), Image.LANCZOS)
    final_arr = np.array(pil_square, dtype=np.float32)

    # 关键1：黑白反转，匹配MNIST白底黑字
    final_arr = 255.0 - final_arr
    # 关键2：和训练完全一致归一化+标准化
    final_arr = final_arr / 255.0
    mean = 0.1307
    std = 0.3081
    final_arr = (final_arr - mean) / std

    # 构造张量 [batch, channel, H, W]
    img_tensor = torch.tensor(final_arr).unsqueeze(0).unsqueeze(0)
    return img_tensor.to(device)

@app.get("/")
def root():
    return {"message": "MNIST数字识别API", "usage": "POST /predict 上传手写数字图片"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        image_bytes = await file.read()
        if not image_bytes:
            raise HTTPException(status_code=400, detail="上传文件为空")

        img_tensor = backend_preprocess(image_bytes)

        with torch.no_grad():
            outputs = model(img_tensor)
            predicted = torch.argmax(outputs, dim=1).item()

        return JSONResponse({
            "prediction": predicted,
            "status": "success"
        })

    except ValueError as ve:
        raise HTTPException(status_code=400, detail=str(ve))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"推理异常：{str(e)}")

# 启动服务
if __name__ =="__main__":#仅当脚本被直接运行时执行（而不是被导入时）
    import uvicorn
    import nest_asyncio
    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()#http://localhost:8000/

模型加载成功


INFO:     Started server process [31724]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:63934 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:63937 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:63938 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:26979 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:26981 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:26983 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:52593 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:52595 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:46885 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [31724]
